## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model =ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

/Users/ayush/Documents/Langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11eab01c0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11ead99f0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content='HI ,MY NAME is ayush and i am chief ai engineer')])

AIMessage(content="Hello Ayush, nice to meet you. It's great to hear that you're a Chief AI Engineer. That's a very impressive title and I'm sure you have a lot of expertise in the field of artificial intelligence. What kind of projects do you usually work on, and what are some of the most interesting challenges you've faced in your role?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 48, 'total_tokens': 120, 'completion_time': 0.22021057, 'completion_tokens_details': None, 'prompt_time': 0.002165923, 'prompt_tokens_details': None, 'queue_time': 0.054080456, 'total_time': 0.222376493}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8a8b-8242-7921-be45-aade3c11003d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 72, 'total_tokens': 120})

In [4]:
# from langchain_core.output_parsers import StrOutputParser
# result = StrOutputParser()
# docs =result.parse(my_model)
# docs.content

In [5]:
from langchain_core.messages import AIMessage
model.invoke(
  [
    HumanMessage(content="HI,My name is Ayush and i am a full stack developer"),
    AIMessage(content="Hello Ayush, nice to meet you. As a Chief AI Engineer, you must be working on some exciting projects, leveraging the power of artificial intelligence to drive innovation and solve complex problems. What specific areas of AI are you currently focused on, such as machine learning, natural language processing, or computer vision"),

    HumanMessage(content="Hey What is my name and what do i do?")
  ]
)

AIMessage(content="Your name is Ayush, and you're a full stack developer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 131, 'total_tokens': 146, 'completion_time': 0.04277787, 'completion_tokens_details': None, 'prompt_time': 0.007141553, 'prompt_tokens_details': None, 'queue_time': 0.056157061, 'total_time': 0.049919423}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8a8b-84e1-7783-b240-af685092f544-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 15, 'total_tokens': 146})

## Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [6]:
!pip install langchain_community

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=ChatMessageHistory()
  return store[session_id]


with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config={"configurable":{"session_id":"chat1"}}

In [9]:
response = with_message_history.invoke(
  [HumanMessage(content="Hi, Myname is ayush and i am full stack developer")],
  config=config
)
response.content

'Hi Ayush, nice to meet you. As a full stack developer, you must have a broad range of skills, from front-end development (client-side) to back-end development (server-side), and possibly even database management. What kind of projects are you currently working on or interested in? Are you using any specific technologies or frameworks like React, Angular, Vue, Node.js, or Django?'

In [10]:
with_message_history.invoke(
  [HumanMessage(content="What is my name?")],
  config=config
)

AIMessage(content='Your name is Ayush.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 143, 'total_tokens': 150, 'completion_time': 0.017753638, 'completion_tokens_details': None, 'prompt_time': 0.00689556, 'prompt_tokens_details': None, 'queue_time': 0.162551883, 'total_time': 0.024649198}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8a8b-8b08-7fd2-9a60-4cc9d65af0b3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 7, 'total_tokens': 150})

In [11]:
## Change the config -->session_id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
  [HumanMessage(content="What's my name")],
  config=config1
)
response.content

"I don't know your name. I'm a large language model, I don't have the ability to recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you and use it in our conversation."

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [12]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
  [
    ("system","you are a helpful assitant,answer the question to the next of your ability"),
    MessagesPlaceholder(variable_name="messages")
  ]
)

chain = prompt|model

In [13]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is ayush")]})

AIMessage(content="Hello Ayush, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 57, 'total_tokens': 84, 'completion_time': 0.051475798, 'completion_tokens_details': None, 'prompt_time': 0.00524759, 'prompt_tokens_details': None, 'queue_time': 0.160299667, 'total_time': 0.056723388}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8a8b-8e17-7dd1-8647-cf93ca140705-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 27, 'total_tokens': 84})

In [14]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [15]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
  [HumanMessage(content="Hi My name is Ayush")],
  config=config
)
response

AIMessage(content="Hello Ayush, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 57, 'total_tokens': 84, 'completion_time': 0.051245673, 'completion_tokens_details': None, 'prompt_time': 0.006157103, 'prompt_tokens_details': None, 'queue_time': 0.159420844, 'total_time': 0.057402776}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8a8b-8f6b-71b2-825c-f168e4a15872-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 27, 'total_tokens': 84})

In [16]:
## Adding more complexity

prompt=ChatPromptTemplate.from_messages(
  [
    ("system","you are a helpful assitant,answer the question to the next of your ability {language}"),
    MessagesPlaceholder(variable_name="messages")
  ]
)

chain = prompt|model

In [17]:
response=chain.invoke(
  {"messages":[HumanMessage(content="Hi I am ayush")],"language":"hindi"}
)
response.content

'नमस्ते आयुष, मैं आपकी मदद के लिए तैयार हूँ। क्या आप किसी विशेष जानकारी या सहायता की तलाश में हैं?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [18]:
with_message_history=RunnableWithMessageHistory(
  chain,
  get_session_history,
  input_messages_key="messages"
)

In [19]:
config={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
  {"messages":[HumanMessage(content="HI I am Ayush")],"language":"hindi"},
  config=config
)
response.content

'नमस्ते आयुष, मैं आपकी कैसे मदद कर सकता हूँ?'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages


In [20]:
!pip install transformers

In [24]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer = trim_messages(
  max_tokens=70,
  strategy="last",
  token_counter=model,
  include_system=True,
  allow_partial=False,
  start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="hi! I'm bob", additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain =(
  RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)|prompt|model
)
response = chain.invoke(
  {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
  }
 
)
response.content

'You like vanilla ice cream.'

In [26]:
## Let's Wrap this in the Message History
with_message_history = RunnableWithMessageHistory(
  chain,
  get_session_history,
  input_messages_key="messages"
)
config={"configurable":{"session_id":"chat5"}}

In [28]:
response = chain.invoke(
  {
    "messages":messages + [HumanMessage(content="What maths problem i ask ?")],
    "language":"English"
  }
)
response.content

'You asked "what\'s 2 + 2" and I answered 4.'